# **Introduction**

Welcome to the **PART 2** of the programming portion of assignment 3. This portion is ONLY REQUIRED for CS 5756, where you will extend Part 1 by implementing the Soft-Actor-Critic (SAC) algorithm (Part 3)

Please read through the following paragraphs carefully.

**Getting Started:** You should complete this assignment on **[Google Colab](https://colab.research.google.com/)**.

**Evaluation:**
See instructions from Part 1.

**Getting Help:** See instructions from Part 1.


# **Setup**

**Please run the cells below to install the necessary packages**.



In [ ]:
import os
import sys
USING_COLAB = 'google.colab' in sys.modules

if USING_COLAB:
  !apt-get -qq update
  !apt-get -qq install -y libosmesa6-dev libgl1-mesa-glx libglfw3 libgl1-mesa-dev libglew-dev patchelf
  !apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
else:
  !pip -q install torch torchvision torchaudio
  !pip -q install numpy tqdm opencv-python matplotlib

!pip -q install -U mediapy renderlab "imageio<3.0" "stable-baselines3[extra]"

# Make robotics env registration robust on local + Colab
if not os.path.exists("Gymnasium-Robotics"):
  !git clone https://github.com/Farama-Foundation/Gymnasium-Robotics.git
!pip -q install -e Gymnasium-Robotics

sys.path.append(os.path.abspath("./Gymnasium-Robotics"))

In [ ]:

import os
# Mujoco GLEW Setup
try:
  if _mujoco_run_once:
    pass
except NameError:
  _mujoco_run_once = False
if not _mujoco_run_once:
  try:
    os.environ['LD_PRELOAD']=os.environ['LD_PRELOAD'] + ':/usr/lib/x86_64-linux-gnu/libGLEW.so'
  except KeyError:
    os.environ['LD_PRELOAD']='/usr/lib/x86_64-linux-gnu/libGLEW.so'
  # presetup so we don't see output on first env initialization
  _mujoco_run_once = True
  if USING_COLAB:
    NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
      with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
          f.write("""{
          "file_format_version" : "1.0.0",
          "ICD" : {
              "library_path" : "libEGL_nvidia.so.0"
          }
      }
      """)
  # Set environment variable to support EGL (off-screen) rendering
  %env MUJOCO_GL=egl

# Import packages and initial seeding


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as D
from stable_baselines3.common.buffers import ReplayBuffer
import torch.optim as optim
import gymnasium as gym
import gymnasium_robotics
import gymnasium.wrappers as wrappers
import random
import tqdm
import matplotlib.pyplot as plt
from torch.distributions import Categorical

In [4]:
seed = 695

# Setting the seed to ensure reproducability
def reseed(seed : int):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

reseed(seed)

In [5]:
# In this block we define wrappers necessary to simplify the environment MDP
def wrap_reach_fixed_goal(env):
  g = np.array([1.486, 0.73, 0.681], dtype=np.float32)
  env.unwrapped._sample_goal = lambda: g
  return env

class FetchRewardWrapper(gym.Wrapper):
  def reset(self, *args, **kwargs):
    obs, info = self.env.reset(*args, **kwargs)
    self.prev_dist = np.linalg.norm(obs['achieved_goal'] - obs['desired_goal'])
    return obs, info

  def step(self, action):
    obs, reward, terminated, truncated, info = self.env.step(action)

    achieved_goal = obs['achieved_goal']
    desired_goal = obs['desired_goal']
    current_dist = np.linalg.norm(achieved_goal - desired_goal)
    reward = -current_dist + (2.0 if info["is_success"] else 0.0)
    self.prev_dist = current_dist
    terminated = terminated or info['is_success']
    return obs, reward, terminated, truncated, info

def make_fetchreach_env(seed: int | None = None):
  env = gym.make("FetchReach-v4", render_mode="rgb_array")
  env = wrap_reach_fixed_goal(env)
  env = FetchRewardWrapper(env)
  env = wrappers.FilterObservation(env, ["desired_goal", "observation"])
  env = wrappers.FlattenObservation(env)
  if seed is not None:
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
  return env

# Visualize Helper Function

Below, we provide the helper function `visualize` for your use. This function will create a visualization of the environment passed in the parameter `env`. If you are using Colab, calling this function will render the visualization within the notebook. If you are using your local machine, this function will instead save a video of the visualization to your current directory (rendering videos in Jupyter Notebooks is not widely supported outside of Colab).

In [6]:
def visualize(env : gym.Env, algorithm=None, video_name="test"):
    """Visualize a policy network for a given algorithm on a single episode

        Args:
            algorithm (PolicyGradient): Algorithm whose policy network will be rolled out for the episode. If
            no algorithm is passed in, a random policy will be visualized.
            video_name (str): Name for the mp4 file of the episode that will be saved (omit .mp4). Only used
            when running on local machine.
    """

    def get_action(obs):
        if not algorithm:
            return env.action_space.sample()
        else:
            return algorithm.compute_action(obs)

    if USING_COLAB:
        import renderlab as rl

        directory = './video'
        env = rl.RenderFrame(env, "output/")
        obs, info = env.reset()
        for i in range(500):
            action = get_action(obs)
            obs, reward, terminated, truncated, info = env.step(action)

            if terminated or truncated:
                break

        env.play()
    else:
        import cv2

        video = cv2.VideoWriter(f"{video_name}.mp4", cv2.VideoWriter_fourcc(*'mp4v'), 24, (600,400))
        obs = env.reset()
        for i in range(500):
            action = get_action(obs)
            obs, reward, terminated, truncated, info = env.step(action)

            if terminated or truncated:
                break

            im = env.render(mode='rgb_array')
            im = im[:,:,::-1]

            video.write(im)

        video.release()
        env.close()
        print(f"Video saved as {video_name}.mp4")

### `PolicyNet` class

Paste below your implementation of the PolicyNet class from A3 Part 1.

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int):
        """Policy network architecture for policy gradient algorithms.

        Args:
            state_dim (int): Dimension of the state space.
            action_dim (int): Dimension of the action space.
            hidden_dim (int): Dimension of the hidden layers.
        """
        super(PolicyNet, self).__init__()
        # TODO: Implement the policy network for the REINFORCE algorithm here


    def forward(self, state: torch.Tensor) -> D.Normal:
        """Forward pass of the policy network.

        Args:
            state (torch.Tensor): State of the environment. Shape (N, state_dim)
        Returns:
            action_dist (D.Normal): Normal distribution representing \pi(a_t | s_t)
        """
        # TODO: Implement the forward pass of the policy network here.
        return NotImplementedError("Make sure you implemented the forward pass of the policy network!")

## **Notes about Fetch Reach Environment**
The environment uses a Fetch Robot, which is a 7-DoF Mobile Manipulator.

The task is a *goal-reaching task*:
The observation space contains '`observation`' which includes the state of the robot in the environment, and '`desired_goal`' which specifies the xyz coordinate that the robot's gripper aims to reach.

See https://robotics.farama.org/envs/fetch/reach/ for more details.


If the goal is reached, `info['is_success']` will be set to 1, and this is an indication that we should terminate the rollout.


The reward is -1 per timestep spent in the environment without completing the task, with 50 steps being the limit (so -50 is the worst episode return).

> Note: For this assignment, we've modified the environment that it only has a fixed goal to reach, and has better reward shaping, since vanilla REINFORCE will produce very noisy gradients for complex markov chains.

**Run the cell below to create and visualize the environment:**

In [ ]:
# Let's initialize the environment first
reseed(seed)
env = make_fetchreach_env(seed=seed)

In [ ]:
visualize(env)


### `PolicyGradient` class

Here, we implement a basic version of the `PolicyGradient` class from A2.

Recall the following methods (that you will be implementing for your algorithms).

- `compute_action(self, state)`: Method that selects an action based on the policy network.

- `compute_loss(self, episode, gamma)`: Method that computes the loss for a given episode.

- `update_policy(self, episodes, optimizer, gamma)`: Method that updates the policy network.

In [ ]:
# No need to implement anything in this cell, just run it.

class PolicyGradient:
    def __init__(
      self,
      policy_net : PolicyNet
    ):
      """Default class for a policy gradient algorithm.

      Args:
          policy_net (PolicyNet): Policy network
      """
      self.policy_net = policy_net

    @property
    def device(self):
      return next(iter(self.policy_net.parameters())).device

    def to(self, device : str | torch.device):
      self.policy_net.to(device)
      return self

    def compute_action(self, state : np.ndarray) -> np.ndarray:
        raise NotImplementedError("Make sure you implemented compute_action()!")

    def compute_loss(
      self,
      episode : list[tuple[
          np.ndarray, np.ndarray, float
      ]],
      gamma : float
    ) -> torch.Tensor:
      raise NotImplementedError("Make sure you implemented compute_loss()!")

    def update_policy(self, episodes, optimizer, gamma):
      raise NotImplementedError("Make sure you implemented update_policy()!")

In [ ]:
def collect_episode(env : gym.Env, policy : PolicyGradient | None = None, seed : int | None = None):
    """
    Run an episode of the environment and return the episode

    Returns:
        episode (list): List of tuples (state, action, reward)
    """
    state, info = env.reset(seed=seed)
    episode = []
    done = False
    while not done:
      if policy is not None:
        action = policy.compute_action(state)
      else:
        action = env.action_space.sample()
      next_state, reward, terminated, truncated, info = env.step(action)
      done = terminated or truncated or info.get('is_success', False)
      episode.append((state, action, reward, next_state, done, info))
      state = next_state
    return episode

def evaluate(env : gym.Env, policy : PolicyGradient | None = None, num_episodes = 100):
    """Evaluate the policy network by running multiple episodes.

    Args:
        env (gym.Env): Environment to evaluate the policy network on
        policy (PolicyGradient): Policy network to evaluate
        num_episodes (int): Number of episodes to run

    Returns:
        average_reward (float): Average total reward per episode
    """
    episode_rewards = []
    for i in range(num_episodes):
      episode = collect_episode(env, policy)
      episode_rewards.append(sum([reward for _, _, reward, _, _, _ in episode]))
    return np.mean(episode_rewards)

### `CriticNetwork` class

Paste below your implementation of the `CriticNetwork` class from A3 Part 1.

In [ ]:
class CriticNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim):
        super().__init__()
        #TODO: Implement the CriticNetwork architecture.

    def forward(self, x, a):
        #TODO: Implement the CriticNetwork forward pass.
        return NotImplementedError()

# (CS5756 ONLY) Part 3: Soft Actor Critic (SAC)

In this part of the assignment, we will implement a version of the Soft Actor Critic (SAC) algorithm. SAC is an off-policy, actor-critic algorithm designed for continuous action spaces. Unlike DDPG, SAC learns a **stochastic** policy and incorporates an **entropy** term in the objective function to encourage exploration. This helps SAC achieve better stability and sample efficiency.

This part requires three steps:
1. Implement `StochasticActor()`
3. Implement `SACAgent()`

Below, we walk you through the steps.

### `StochasticActor` class

The `StochasticActor` class should define a neural network that takes in a state, and outputs an action sampled stochastically from a distribution. The network should have the following architecture:

- Input layer: a fully-connected layer with `state_dim` input nodes and `hidden_dim` output nodes, followed by a ReLU activation function.

- Intermediate layer: a fully-connected layer with `hidden_dim` input nodes and `hidden_dim` output nodes, followed by a ReLU activation function.

- Output layer I: a fully-connected layer with `hidden_dim` input nodes and `action_dim` output nodes for $\mu(s_t)$.

- Output layer II: a fully-connected layer with `hidden_dim` input nodes and `action_dim` output nodes for $\log(\sigma(s_t))$).

We again use the $\tanh$ squashing trick from our DDPG implementation, where we apply $\tanh$ to the neural network's output to constrain it to the range $[-1, 1]$. In the forward function, we do two different things:

For output layer I, we simply output the mean.

For output layer II, we want to scale the `log_std` distribution from $[-1, 1]$ to $[\text{LOG_STD_MIN}, \text{LOG_STD_MAX}]$. Use similar tricks from your DDPG implementation.

For `sample_action_distribution(self, x)`, we sample an initial $x_t$ from a `torch.distributions.normal.Normal` object with the proper $\mu(s_t)$ and $\sigma(s_t)$. Then, we apply the $\tanh$ squashing + biasing trick to our action $y_t$ to get our action in the range $[\text{action_low}, \text{action_high}]$.

To compute the log probability of our action, we take the log probability of our `torch.distributions.normal.Normal` at the sampled $x_t$, and subtract a corrective factor of $\log(\text{action_scale} * (1 - y_t^2) + 1e^{-6})$ to account for the change in probability density from our $\tanh$ squashing trick.

The `sample_action_distribution(self, x)` mmethod should then output the scaled output action $y_t$, and the corrected log probability.

In [ ]:
LOG_STD_MAX = 2
LOG_STD_MIN = -5

class StochasticActor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim, action_high, action_low):
        super().__init__()
        # TODO: Implement the StochasticActor network architecture.

    def forward(self, x):
        # TODO: Implement the StochasticActor forward pass.
        return NotImplementedError()

    def sample_action_distribution(self, x):
        # TODO: Implement sampling from the action distribution.
        return NotImplementedError()

### `SACAgent` class

The `SACAgent` class should define the SAC algorithm. Recall that SAC is an **off-policy algorithm**, so it trains very similarly to DDPG.

The basic structure of the `DDPGAgent` consists of:
- TWO `CriticNetwork()` and TWO target `CriticNetwork()`
- One `DeterministicActor()`.

We use two critic networks in SAC to help reduce overestimation bias.

The formulation of SAC is as follows:

Suppose we sampled a batch of transitions of form $(s, s', a, r, d, i)$:

We then want to compute the **target q-values** like so, denoting $a' \sim \mu_\theta(s')$:

$$y(r, s', d) = r + \gamma (1-d)[\min(Q_{targ, 1}(s', a'), Q_{targ, 2}(s', a')) - \alpha\log\pi_\theta(a'|s')]$$

where $\alpha$ is the temperature parameter.

The **critic** is then updated by one step of gradient descent like:

$$∇\frac{1}{|B|}∑_{(s, s', a, r, d) \in B}(Q(s, a)-y(r, s', d))^2$$

The actor is updated by computing a loss dependent on entropy (which encourages exploration) and the Q-value of the actions taken by the actor (denoting $a = \mu_\theta(s)$).

$$∇\frac{1}{|B|}∑_{s \in B}[\alpha\log\pi_\theta(a|s) - \min(Q_1(s, a), Q_2(s, a))]$$

We update the parameters of the target network according to a rate $\tau$; the update step is:

$$\theta_{target, 1} = (1-\tau)\theta_{target, 1} + \tau\theta_1$$
$$\theta_{target, 2} = (1-\tau)\theta_{target, 2} + \tau\theta_2$$

And finally, we want to taper the value of $\alpha$ over the course of the training, to strategically move from exploration -> exploitation. We do this by treating $\alpha$ as a learnable parameter, and have the following loss function:

$$∇\frac{1}{|B|}∑_{s \in B}[-\log(\alpha) * (\log\pi_\theta(a|s) + \alpha_{target})]$$

Note: you should use the learnable `log_alpha` parameter here.

We want you to implement the following methods:

- `train(self, num_iterations, batch_size)`: This is the method that trains the actor and critic in the SAC algorithm. It should follow this basic format:
  - For each iteration in num_iterations, you should step through the environment, and add the transition to your replay buffer. Once you have accumulated `learning_starts` transitions, call the `update_policy()` method every step.

  - Print out some useful training information, and track your losses.

  - **IMPORTANT**: When stepping through the environment, don't forget to set obs = next_obs when you are finished with the step, as well as reset the environment when it terminates!

  - **NOTE**: We use a separate `eval_env` to run evaluations, as we do not want to corrupt the environment between evaluation and data collection steps.

- `compute_action(self, state)`: Compute the action using the `sample_action_distribution` method of the Actor.

- `compute_critic_loss(self, data)`: Method that computes the loss function for your critic networks.

- `compute_actor_loss(self, data)`: Method that computes the loss function for your actor network. **You should also return the log probs to be used in your alpha loss.**

- `update_target_net(self)`: Method that performs the weighted target net update.

- `update_actor(self, loss)`: Take in a loss computed by `compute_actor_loss(self, data)`, and update the actor network.

- `update_actor(self, loss1, loss2)`: Take in losses computed by `compute_critic_loss(self, data)`, and update the critic networks.

- `update_alpha(self, loss)`: Take in a loss computed by `compute_alpha_loss(self, log_probs)`, and update the value of `log_alpha`.

- `update_policy(self, batch_size)`: Method that updates the actor and critic using batch of transitions. It should follow this basic format:
  - Sample `batch_size` transitions from your replay buffer.
  - Call `update_critic()`.
  - Call `update_actor()`.
  - Call `update_alpha()`.
  - Call `update_target_net()`.


**Important**: Please do not deviate from the `SACAgent()` format/method structure, it makes it much easier for us to help you in office hours!

In [ ]:
import copy

class SACAgent():
    def __init__(
        self,
        env,
        gamma,
        tau,
        alpha,
        actor_lr,
        critic_lr,
        network_hidden_dim,
        learning_starts=5000,
        eval_interval=500,
        eval_episodes=10,
        eval_env=None,
    ):
        self.env = env
        self.eval_env = eval_env if eval_env is not None else make_fetchreach_env()
        self.device = "cpu" if not torch.cuda.is_available() else "cuda"

        obs_dim = np.prod(self.env.observation_space.shape)
        act_dim = np.prod(self.env.action_space.shape)

        self.actor = StochasticActor(obs_dim, act_dim, network_hidden_dim, self.env.action_space.high, self.env.action_space.low).to(self.device)
        self.critic1 = CriticNetwork(obs_dim, act_dim, network_hidden_dim).to(self.device)
        self.critic2 = CriticNetwork(obs_dim, act_dim, network_hidden_dim).to(self.device)
        self.critic1_target = CriticNetwork(obs_dim, act_dim, network_hidden_dim).to(self.device)
        self.critic2_target = CriticNetwork(obs_dim, act_dim, network_hidden_dim).to(self.device)
        self.critic1_target.load_state_dict(self.critic1.state_dict())
        self.critic2_target.load_state_dict(self.critic2.state_dict())

        self.replay_buffer = ReplayBuffer(
            int(1e6),
            self.env.observation_space,
            self.env.action_space,
            self.device,
            handle_timeout_termination=False,
        )
        self.critic_optimizer = optim.Adam(list(self.critic1.parameters()) + list(self.critic2.parameters()), lr=critic_lr)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)

        self.log_alpha = torch.tensor(np.log(alpha), requires_grad=True, device=self.device)
        self.target_entropy = -act_dim
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=actor_lr)

        self.tau = tau
        self.gamma = gamma
        self.learning_starts = learning_starts
        self.eval_interval = eval_interval
        self.eval_episodes = eval_episodes

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def compute_action(self, obs):
        # TODO: Implement action computation.

    def train(self, num_iterations, batch_size):
        # TODO: Implement training loop.

    def compute_critic_loss(self, data):
        # TODO: Implement critic loss computation.

    def compute_actor_loss(self, data):
        # TODO: Implement actor loss computation.

    def compute_alpha_loss(self, log_probs):
        # TODO: Implement alpha loss computation.

    def update_critic(self, loss1, loss2):
        # TODO: Implement critic update step.

    def update_actor(self, loss):
        # TODO: Implement actor update step.

    def update_target_net(self):
        # TODO: Implement target update step.

    def update_alpha(self, loss):
        # TODO: Implement alpha update step.

    def update(self, step, num_iterations, batch_size):
        # TODO: Implement general update step (which calls other methods). Don't forget to print/return some useful information here.

Now that you have implemented SAC, it is time to train and evaluate a policy. Below, we provide training hyperparameters which you should use in your experiments.

We encourage you to modify the initial values of `alpha` and `learning_starts` both positively and negatively to get a sense of how they affect training. You do not have to justify your choice.

* Expected Training Time (Colab CPU): < 5 minutes
* Expected Episodic Reward: 1.5

In [ ]:
# Feel free to use the space below to run experiments and create plots used in your writeup.
reseed(seed)
env = make_fetchreach_env(seed=seed)
eval_env = make_fetchreach_env(seed=seed + 1)

sac_agent = SACAgent(
    env=env,
    gamma=0.98,
    tau=0.002,
    alpha=0.2,
    actor_lr=3e-4,
    critic_lr=3e-4,
    network_hidden_dim=256,
    learning_starts=5000,
    eval_env=eval_env,
)

losses = sac_agent.train(15000, 256)

final_rew = evaluate(eval_env, sac_agent, 20)
print(f"Average reward: {final_rew:.4f}")

visualize(eval_env, algorithm=sac_agent, video_name="sac")

## Please submit two separate plots in your writeup, of the value and policy loss across training. As well, note your average reward.

In [ ]:
# Feel free to use this space for plotting.